<a href="https://colab.research.google.com/github/guillaumevalette2-hash/mse_gh/blob/main/PGT/pgt_auc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Cloud - parametres

In [ ]:
import numpy as np
from itertools import product, combinations_with_replacement
from math import factorial
from collections import Counter
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import roc_auc_score
from sklearn.svm import SVC
from sklearn.kernel_ridge import KernelRidge
from sklearn.model_selection import GridSearchCV
from scipy.optimize import minimize, LinearConstraint

# ══════════════════════════════════════════════════════════════════════════════
# PARAMÈTRES
# ══════════════════════════════════════════════════════════════════════════════
params = {
    # ── données MNIST ────────────────────────────────────────────────────────
    "neg_digits":  [3, 5],
    "pos_digits":  [8],
    "img_size":    4,          # 28×28 -> 4×4  (dim ambiante 16) ; 7 -> dim 49
    "n_train":     300,
    "n_test":      1000,
    "n_unlabeled": 3000,
    "seed":        129785,
    "seeds":       {42: 73659, 43: 96778},

    # ── QP ambient (Sobolev) ─────────────────────────────────────────────────
    "weights":     {0: 0, 1: 1.0, 2: 0.5, 3: 0.0},   # w3=0 par défaut
    "lambda_G":    1e-9,
    "qp_margin":   1.0,
    "thres1":      1e-3,
    "thres2":      1e6,
    "const_pen":   1e-5,
    "n_G":         600,
    "batch_size":  200,
    "deg_start":   2,          # degré du QP ambiant

    # ── plongement spectral (point 2) ────────────────────────────────────────
    "n_layers":    1,
    "embed_dim":   152,         # nombre de VP retenus par couche
    "k_proj":      50,          # taille des blocs de projection orthogonale
    "n_deg":       2,          # degré des monômes dans chaque bloc projeté
    "lambda_G_emb": 1e-9,      # régularisation Gram du plongement
    "emb_thres1":  1e-3,       # plancher relatif (s / s_max)
    "emb_thres2":  2.0,       # plafond relatif
    "weights_plg": {0: 0, 1: 1.0, 2: 0.5, 3: 0.0},  # Sobolev du plongement
    "activation":  "ReLu",
}


# ══════════════════════════════════════════════════════════════════════════════
# DONNÉES
# ══════════════════════════════════════════════════════════════════════════════
from tensorflow.keras.datasets import mnist

def load_mnist_binary(neg_list, pos_list, img_size,
                      n_train, n_test, n_unlabeled, seed=params["seeds"][42]):
    (Xtr, ytr), (Xte, yte) = mnist.load_data()
    X64 = np.concatenate([Xtr, Xte], axis=0).astype(float)
    y_raw = np.concatenate([ytr, yte], axis=0)

    keep = set(neg_list) | set(pos_list)
    mask = np.isin(y_raw, list(keep))
    X64, y_raw = X64[mask], y_raw[mask]
    y = np.where(np.isin(y_raw, list(pos_list)), 1.0, -1.0)

    if 28 % img_size != 0:
        raise ValueError("img_size doit diviser 28")
    b = 28 // img_size
    Xs = X64.reshape(-1, img_size, b, img_size, b).mean(axis=(2, 4))
    Xs = Xs.reshape(-1, img_size * img_size)
    Xs = (Xs - Xs.mean(0)) / (Xs.std(0) + 1e-8)

    rng = np.random.default_rng(seed)
    idx_neg = np.where(y < 0)[0]
    idx_pos = np.where(y > 0)[0]
    rng.shuffle(idx_neg)
    rng.shuffle(idx_pos)

    n_tr_c = n_train // 2
    n_te_c = n_test // 2
    tr = np.concatenate([idx_neg[:n_tr_c], idx_pos[:n_tr_c]])
    te = np.concatenate([idx_neg[n_tr_c:n_tr_c + n_te_c],
                         idx_pos[n_tr_c:n_tr_c + n_te_c]])
    used = set(tr) | set(te)
    unlab_pool = np.array([i for i in range(len(Xs)) if i not in used])
    unlab = (rng.choice(unlab_pool, n_unlabeled, replace=False)
             if len(unlab_pool) > n_unlabeled else unlab_pool)

    rng.shuffle(tr); rng.shuffle(te); rng.shuffle(unlab)
    return Xs[tr], y[tr], Xs[te], y[te], Xs[unlab], y[unlab]




# ══════════════════════════════════════════════════════════════════════════════
# QP SOBOLEV
# ══════════════════════════════════════════════════════════════════════════════
def solve_qp(A, y, G, margin):
    n, k = A.shape
    Gr = G + 1e-12 * np.eye(k)
    con = LinearConstraint(np.diag(y) @ A, lb=margin, ub=np.inf)
    try:
        c0 = np.linalg.lstsq(A, 1.5 * margin * y, rcond=None)[0]
    except Exception:
        c0 = np.zeros(k)
    res = minimize(lambda c: c @ Gr @ c, c0, jac=lambda c: 2 * Gr @ c,
                   constraints=[con], method='SLSQP',
                   options={'maxiter': 500, 'ftol': 1e-11})
    c = res.x
    marge_eff = float(np.min(y * (A @ c)))
    return c, marge_eff >= margin - 1e-4, marge_eff


def _multiplicity(combo):
    c = Counter(combo)
    m = factorial(len(combo))
    for v in c.values():
        m //= factorial(v)
    return m


def poly_eval_from_powers(U, powers):
    n, nf = U.shape[0], powers.shape[0]
    Phi = np.ones((n, nf))
    for a in range(U.shape[1]):
        e = powers[:, a]
        if not np.any(e):
            continue
        maxe = int(e.max())
        pw = np.empty((maxe + 1, n))
        pw[0] = 1.0
        if maxe >= 1:
            pw[1] = U[:, a]
        for p in range(2, maxe + 1):
            pw[p] = pw[p - 1] * U[:, a]
        Phi *= pw[e].T
    return Phi


class MonomialCache:
    def __init__(self, U_G):
        self.U_G = U_G
        self.cache = {}
    def eval(self, alpha):
        key = tuple(int(x) for x in alpha)
        if key not in self.cache:
            v = np.ones(self.U_G.shape[0])
            for d, e in enumerate(key):
                if e:
                    v = v * self.U_G[:, d] ** e
            self.cache[key] = v
        return self.cache[key]


def _reduce_monomial(p, combo_count):
    coeff = 1.0
    r = p.copy()
    for a, k in combo_count.items():
        if r[a] < k:
            return 0.0, None
        for i in range(k):
            coeff *= (r[a] - i)
        r[a] -= k
    return coeff, r


def build_master_grams_moments(U_G, powers_master, orders_needed):
    d = U_G.shape[1]
    deg_max = int(np.max(np.sum(powers_master, axis=1)))
    max_mom_deg = 2 * deg_max

    print(f"  précalcul des moments jusqu'au degré {max_mom_deg} ...")
    M = precompute_moments_dict(U_G, max_mom_deg)

    master = {}

    for o in sorted(orders_needed):
        weights_o = {o: 1.0}
        master[o] = sobolev_gram_from_moments(
            powers_master,
            M,
            weights_o,
            d
        )

    # moyenne des monômes sur le même cloud U_G
    mean0 = np.array([
        M.get(tuple(int(x) for x in alpha), 0.0)
        for alpha in powers_master
    ])

    return master, mean0


def gram_from_master(master, weights, idx_rows, idx_cols):
    G = np.zeros((len(idx_rows), len(idx_cols)))
    for o, wo in weights.items():
        if wo and o in master:
            G += wo * master[o][np.ix_(idx_rows, idx_cols)]
    return G


def fit_qp_from_master(X_tr, y_tr, X_te, y_te, powers_master, idx, lo, span,
                       master, mean0, weights, lambda_G, thres1, thres2,
                       const_pen, qp_margin, titre=""):
    n = X_tr.shape[0]
    powers = powers_master[idx]
    def to_u(X):
        return 2 * (X - lo) / span - 1.0
    Phi_tr = poly_eval_from_powers(to_u(X_tr), powers)

    G = gram_from_master(master, weights, idx, idx) + lambda_G * np.eye(len(idx))
    mean_phi = mean0[idx] if mean0 is not None else np.zeros(len(idx))

    s_g, V_g = np.linalg.eigh(G)
    keep = (s_g > thres1) & (s_g < thres2)
    if keep.sum() == 0:
        raise ValueError("bande spectrale vide")
    T = V_g[:, keep] / np.sqrt(s_g[keep])
    r = int(keep.sum())
    A_qp = np.hstack([(Phi_tr - mean_phi) @ T, np.ones((n, 1))])
    G_qp = np.eye(r + 1)
    G_qp[-1, -1] = const_pen
    sol, feas, marge = solve_qp(A_qp, y_tr, G_qp, qp_margin)
    coef = T @ sol[:r]
    off = sol[-1] - mean_phi @ coef

    f_te = poly_eval_from_powers(to_u(X_te), powers) @ coef + off
    f_tr = Phi_tr @ coef + off
    mse_te = float(np.mean((f_te - y_te) ** 2))
    normH = float(np.sqrt(max(sol[:r] @ sol[:r], 0)))

    def acc(f, y):
        sg = np.sign(f)
        return float(np.mean(np.where(sg == 0, 0.5, (sg == np.sign(y)).astype(float))))

    thr = 0.5 * (f_tr[y_tr > 0].mean() + f_tr[y_tr < 0].mean())
    def acc_thr(f, y, t):
        sg = np.sign(f - t)
        return float(np.mean(np.where(sg == 0, 0.5, (sg == np.sign(y)).astype(float))))
    acc_te_thr = acc_thr(f_te, y_te, thr)
    try:
        auc = roc_auc_score((y_te > 0).astype(int), f_te)
    except Exception:
        auc = float('nan')
    print(f"  [{titre}] n_feat={len(idx)} rang={r} | faisable={feas} marge={marge:.3f} "
          f"‖u‖_H={normH:.4f} MSE_te={mse_te:.4f} | "
          f"acc_tr={acc(f_tr, y_tr):.3f} acc_te={acc(f_te, y_te):.4f} "
          f"acc_te_seuil={acc_te_thr:.4f} (thr={thr:.4f}) AUC={auc:.4f}")
    return {
        'coef': coef, 'off': off, 'mse_te': mse_te, 'normH': normH,
        'acc_te': acc(f_te, y_te), 'acc_te_thr': acc_te_thr, 'thr': thr, 'auc': auc
    }



# plongement spectral

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PLONGEMENT SPECTRAL SANS BLOCS
#   - monômes de degré 1..n_deg dans tout l'espace
#   - Gram Sobolev par dictionnaire de moments
#   - on ne calcule un ordre de dérivation que si le poids correspondant est > 0
#   - sélection des VP : bande spectrale + AUC sur le train
# ══════════════════════════════════════════════════════════════════════════════

def activate(X, mode):
    if mode == "tanh":
        return np.tanh(X)
    if mode == "sigmoid":
        return 1 / (1 + np.exp(-X))
    return X


def poly_features_full(X, n_deg):
    """Monômes de degré 1..n_deg (sans constante). Retourne Phi et la liste des multi-indices."""
    poly = PolynomialFeatures(degree=n_deg, include_bias=False)
    Phi = poly.fit_transform(X)
    return Phi, poly.powers_


def precompute_moments_dict(X, max_deg):
    """M[alpha] = mean(x^alpha), |alpha| <= max_deg."""
    n, d = X.shape
    M = {}

    def compositions(total, dim, prefix=()):
        if dim == 1:
            yield prefix + (total,)
            return
        for k in range(total + 1):
            yield from compositions(
                total - k,
                dim - 1,
                prefix + (k,)
            )

    for deg in range(max_deg + 1):
        for alpha in compositions(deg, d):
            col = np.ones(n)
            for j, e in enumerate(alpha):
                if e:
                    col *= X[:, j] ** e
            M[alpha] = float(col.mean())

    return M
def build_qp_gram_from_moments(X_cloud, powers, weights, lambda_G):
    """
    Gram Sobolev exact par moments pour le QP ambient.
    Même convention que sobolev_gram_from_moments.
    """
    d = X_cloud.shape[1]
    max_mom_deg = 2 * int(np.max(np.sum(powers, axis=1)))

    print(f"  moments QP jusqu'au degré {max_mom_deg} ...")
    M = precompute_moments_dict(X_cloud, max_mom_deg)

    G = sobolev_gram_from_moments(
        powers, M, weights, d
    )

    G += lambda_G * np.eye(len(powers))
    return G
def build_master_grams_moments(U_G, powers_master, orders_needed):
    """
    Même objet que build_master_grams, mais calculé exactement
    par moments sur le même cloud U_G.
    """
    d = U_G.shape[1]
    deg_max = int(np.max(np.sum(powers_master, axis=1)))

    max_mom_deg = 2 * deg_max
    M = precompute_moments_dict(U_G, max_mom_deg)

    master = {}

    for o in sorted(orders_needed):
        weights_o = {o: 1.0}
        master[o] = sobolev_gram_from_moments(
            powers_master, M, weights_o, d
        )

    # exactement le même centrage que l'ancien master
    mean0 = np.array([
        M.get(tuple(int(x) for x in alpha), 0.0)
        for alpha in powers_master
    ])

    return master, mean0

def sobolev_gram_from_moments(powers, M, weights, d):
    """
    Gram Sobolev sur la base de monômes donnée par `powers`.
    On ne construit les termes d'ordre o que si weights[o] > 0.
    powers : array (n_feat, d) d'exposants (sans la constante).
    """
    nf = powers.shape[0]
    alphas = [tuple(int(x) for x in powers[i]) for i in range(nf)]
    G = np.zeros((nf, nf))

    def mom(gamma):
        if any(x < 0 for x in gamma):
            return 0.0
        return M.get(tuple(gamma), 0.0)

    # --- ordre 0 (L²) ---
    w0 = weights.get(0, 0.0)
    if w0:
        for i in range(nf):
            ai = alphas[i]
            for j in range(i, nf):
                aj = alphas[j]
                gamma = tuple(ai[k] + aj[k] for k in range(d))
                val = mom(gamma)
                G[i, j] = val
                G[j, i] = val
        G *= w0

    # --- ordre 1 (gradient) ---
    w1 = weights.get(1, 0.0)
    if w1:
        Gg = np.zeros((nf, nf))
        for dim in range(d):
            for i in range(nf):
                ei = alphas[i][dim]
                if ei == 0:
                    continue
                for j in range(i, nf):
                    ej = alphas[j][dim]
                    if ej == 0:
                        continue
                    gamma = [alphas[i][k] + alphas[j][k] for k in range(d)]
                    gamma[dim] -= 2
                    val = ei * ej * mom(gamma)
                    Gg[i, j] += val
                    if i != j:
                        Gg[j, i] += val
        G += w1 * Gg

    # --- ordre 2 (Hessien / dérivées secondes) ---
    w2 = weights.get(2, 0.0)
    if w2:
        Gh = np.zeros((nf, nf))
        # dérivées ∂_a ∂_b
        for a in range(d):
            for b in range(a, d):  # symétrie
                mult = 1.0 if a == b else 2.0  # on compte une seule fois les a≠b
                for i in range(nf):
                    # coefficient de ∂_a ∂_b (x^{α_i})
                    ci = alphas[i][a]
                    if a == b:
                        if ci < 2:
                            continue
                        ci = ci * (ci - 1)
                    else:
                        if ci == 0 or alphas[i][b] == 0:
                            continue
                        ci = ci * alphas[i][b]
                    for j in range(i, nf):
                        cj = alphas[j][a]
                        if a == b:
                            if cj < 2:
                                continue
                            cj = cj * (cj - 1)
                        else:
                            if cj == 0 or alphas[j][b] == 0:
                                continue
                            cj = cj * alphas[j][b]
                        gamma = [alphas[i][k] + alphas[j][k] for k in range(d)]
                        gamma[a] -= 1 + (1 if a == b else 0)
                        gamma[b] -= 1
                        # correction : on a retiré 2 pour a==b ou 1+1 pour a≠b
                        if a == b:
                            gamma[a] = alphas[i][a] + alphas[j][a] - 2
                        else:
                            gamma[a] = alphas[i][a] + alphas[j][a] - 1
                            gamma[b] = alphas[i][b] + alphas[j][b] - 1
                        val = ci * cj * mom(tuple(gamma))
                        Gh[i, j] += mult * val
                        if i != j:
                            Gh[j, i] += mult * val
        G += w2 * Gh

    return G


def _auc_score_1d(scores, y):
    """max(AUC, 1-AUC) — invariant au signe."""
    try:
        a = roc_auc_score((y > 0).astype(int), scores)
        return max(a, 1.0 - a)
    except Exception:
        return 0.5


def poly_embedding(X_input, X_cloud, X_tr, y_tr, weights, lambda_G,
                   n_deg, embed_dim, emb_thres1, emb_thres2):
    """
    Plongement spectral SANS blocs.
      1. monômes de degré 1..n_deg sur tout l'espace
      2. Gram Sobolev par moments (dict) — seuls les ordres avec poids > 0 sont calculés
      3. bande spectrale
      4. parmi les VP de la bande, on garde les embed_dim de meilleur AUC train
    """
    d = X_cloud.shape[1]
    print(f"  monômes deg 1..{n_deg} dans R^{d}")

    Phi_cloud, powers = poly_features_full(X_cloud, n_deg)
    Phi_tr, _ = poly_features_full(X_tr, n_deg)
    Phi_in, _ = poly_features_full(X_input, n_deg)
    n_feat = powers.shape[0]
    print(f"  n_feat = {n_feat}")

    # moments nécessaires : jusqu'à 2*n_deg (pour les produits de monômes)
    # + éventuellement +2 si w2 > 0, mais 2*n_deg suffit déjà pour deg≤2 et w2
    max_mom_deg = 2 * n_deg
    print(f"  précalcul des moments jusqu'au degré {max_mom_deg} ...")
    M = precompute_moments_dict(X_cloud, max_mom_deg)
    print(f"  {len(M)} moments non nuls (potentiels)")

    G = sobolev_gram_from_moments(powers, M, weights, d)
    G += lambda_G * np.eye(n_feat)

    eigvals, eigvecs = np.linalg.eigh(G)
    print(
    f"  spectre brut : "
    f"min={eigvals.min():.6g} "
    f"max={eigvals.max():.6g} "
    f"median={np.median(eigvals):.6g}"
    )
    print(
    "  VP < 1 :",
    np.sum(eigvals < 1.0),
    " | VP < 0.5 :",
    np.sum(eigvals < 0.5),
    " | VP < 0.1 :",
    np.sum(eigvals < 0.1)
    )
    order = np.argsort(eigvals)[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]

    smax = max(float(eigvals[0]), 1e-30)
    in_band = np.where((eigvals > emb_thres1 * smax) & (eigvals < emb_thres2 * smax))[0]
    if len(in_band) == 0:
        raise ValueError(f"aucune VP dans la bande [{emb_thres1}, {emb_thres2}]")

    # AUC sur le train
    auc_scores = np.array([
        _auc_score_1d(Phi_tr @ eigvecs[:, idx], y_tr) for idx in in_band
    ])
    order_auc = np.argsort(-auc_scores)
    n_keep = min(embed_dim, len(in_band))
    chosen = order_auc[:n_keep]
    idx_sel = in_band[chosen]
    L = eigvals[idx_sel]
    V = eigvecs[:, idx_sel]

    print(f"  spectre : {len(eigvals)} VP | dans la bande : {len(in_band)} | "
          f"retenues (top AUC train) : {n_keep}")
    print(f"  VP gardées (λ) : {np.array2string(L, precision=4, max_line_width=100)}")
    print(f"  AUC train      : {np.array2string(auc_scores[chosen], precision=4, max_line_width=100)}")

    V_norm = V / np.sqrt(np.maximum(L, 1e-30))[None, :]
    embed_means = (Phi_cloud @ V_norm).mean(axis=0)
    return Phi_in @ V_norm - embed_means, V_norm, embed_means, powers

#QP

In [ ]:




# ══════════════════════════════════════════════════════════════════════════════
# EXPÉRIENCE
# ══════════════════════════════════════════════════════════════════════════════
X_train, y_train, X_test, y_test, X_unlab, y_unlab = load_mnist_binary(
    params["neg_digits"], params["pos_digits"], params["img_size"],
    params["n_train"], params["n_test"], params["n_unlabeled"], seed=params["seed"])
X_all = np.vstack([X_train, X_unlab])
y_all = np.concatenate([y_train, y_unlab])   # labels disponibles pour sélection AUC / diag
d = X_train.shape[1]
print(f"digits {params['neg_digits']} vs {params['pos_digits']}, "
      f"{params['img_size']}×{params['img_size']} -> dim {d}")
print(f"  train: {len(X_train)}  test: {len(X_test)}  unlabel`ed: {len(X_unlab)}")

# ── QP ambient ───────────────────────────────────────────────────────────────
poly0 = PolynomialFeatures(degree=params["deg_start"], include_bias=False)
poly0.fit(np.zeros((1, d)))
powers_master = poly0.powers_
idx_active = list(range(powers_master.shape[0]))

print(
    f"\nPool maître : {powers_master.shape[0]} monômes "
    f"(actifs deg<={params['deg_start']})"
)

lo0 = X_all.min(axis=0)
hi0 = X_all.max(axis=0)
span0 = np.where(hi0 - lo0 > 1e-12, hi0 - lo0, 1.0)

# Même cloud de Gram qu'avant : n_G points
rng_g0 = np.random.default_rng(params["seeds"][43])
idx_g0 = rng_g0.choice(
    len(X_all),
    min(params["n_G"], len(X_all)),
    replace=False
)

U_G0 = 2.0 * (X_all[idx_g0] - lo0) / span0 - 1.0

orders_needed = {
    o for o in [0, 1, 2, 3]
    if params["weights"].get(o, 0.0)
}

print(
    f"Construction du Gram maître PAR MOMENTS "
    f"pour ordres {sorted(orders_needed)} ..."
)

master, mean0 = build_master_grams_moments(
    U_G0,
    powers_master,
    orders_needed
)

print("Gram maître prêt.")

print("\n" + "=" * 72)
print(f"QP AMBIANT deg<={params['deg_start']}")
print("=" * 72)

sol0 = fit_qp_from_master(
    X_train, y_train,
    X_test, y_test,
    powers_master,
    idx_active,
    lo0,
    span0,
    master,
    mean0,
    params["weights"],
    params["lambda_G"],
    params["thres1"],
    params["thres2"],
    params["const_pen"],
    params["qp_margin"],
    titre=f"QP deg<={params['deg_start']}"
)

# ── plongement spectral (point 2) ────────────────────────────────────────────
print("\n" + "=" * 72)
print(f"PLONGEMENT SPECTRAL ({params['n_layers']} couche(s), "
      f"embed_dim={params['embed_dim']}, k_proj={params['k_proj']}, "
      f"n_deg={params['n_deg']}, bande=[{params['emb_thres1']:g},{params['emb_thres2']:g}])")
print("=" * 72)

# centrage comme dans un_moment
center = X_all.mean(axis=0)
Xc_train = X_train - center
Xc_test = X_test - center
Xc_all = X_all - center

rng = np.random.default_rng(0)
E_train, E_test, E_all = Xc_train, Xc_test, Xc_all

layer_results = []
for layer in range(params["n_layers"]):
    print(f"── couche {layer + 1} ──")
    E_all_new, V_norm, means, powers = poly_embedding(
        E_all, E_all, E_train, y_train,
        params["weights_plg"], params["lambda_G_emb"],
        params["n_deg"], params["embed_dim"],
        params["emb_thres1"], params["emb_thres2"])

    def embed_new(Xin, V=V_norm, m=means, nd=params["n_deg"]):
        Phi, _ = poly_features_full(Xin, nd)
        return activate(Phi @ V - m, params["activation"])

    E_train = embed_new(E_train)
    E_test  = embed_new(E_test)
    E_all   = activate(E_all_new, params["activation"])
    print(f"  dims plongées : {E_all.shape[1]}")

        # ── diagnostics de séparation à cette couche ────────────────────────────
    print("\n  " + "-" * 68)
    print(f"  DIAGNOSTICS COUCHE {layer + 1}")
    print("  " + "-" * 68)

    # QP linéaire dans l'espace plongé
    G_lin = np.eye(E_train.shape[1])
    A_lin = E_train

    sol_lin, feas_lin, marge_lin = solve_qp(
        A_lin, y_train, G_lin, params["qp_margin"]
    )

    f_tr_lin = E_train @ sol_lin
    f_te_lin = E_test @ sol_lin

    acc_tr_lin = float(
        np.mean(np.sign(f_tr_lin) == np.sign(y_train))
    )
    acc_te_lin = float(
        np.mean(np.sign(f_te_lin) == np.sign(y_test))
    )

    try:
        auc_lin = roc_auc_score(
            (y_test > 0).astype(int),
            f_te_lin
        )
    except Exception:
        auc_lin = float("nan")

    # SVM RBF dans l'espace plongé
    param_grid_svm_emb = {
        "C": [0.1, 1, 10, 100],
        "gamma": ["scale", 0.01, 0.1, 1]
    }

    svm_emb = GridSearchCV(
        SVC(kernel="rbf"),
        param_grid_svm_emb,
        cv=5,
        n_jobs=-1
    )
    svm_emb.fit(E_train, y_train)

    f_svm_emb = svm_emb.decision_function(E_test)

    acc_svm_emb = float(
        np.mean(np.sign(f_svm_emb) == np.sign(y_test))
    )

    try:
        auc_svm_emb = roc_auc_score(
            (y_test > 0).astype(int),
            f_svm_emb
        )
    except Exception:
        auc_svm_emb = float("nan")

    # Ridge RBF dans l'espace plongé
    param_grid_kr_emb = {
        "alpha": [1e-3, 1e-2, 1e-1, 1.0],
        "gamma": [0.001, 0.01, 0.1, 1]
    }

    kr_emb = GridSearchCV(
        KernelRidge(kernel="rbf"),
        param_grid_kr_emb,
        cv=5,
        n_jobs=-1
    )
    kr_emb.fit(E_train, y_train)

    f_kr_emb = kr_emb.predict(E_test)

    acc_kr_emb = float(
        np.mean(np.sign(f_kr_emb) == np.sign(y_test))
    )

    try:
        auc_kr_emb = roc_auc_score(
            (y_test > 0).astype(int),
            f_kr_emb
        )
    except Exception:
        auc_kr_emb = float("nan")

    layer_results.append({
        "layer": layer + 1,
        "dim": E_train.shape[1],
        "qp_acc": acc_te_lin,
        "qp_auc": auc_lin,
        "svm_acc": acc_svm_emb,
        "svm_auc": auc_svm_emb,
        "ridge_acc": acc_kr_emb,
        "ridge_auc": auc_kr_emb,
        "qp_margin": marge_lin
    })

    print(
        f"  QP linéaire : acc={acc_te_lin:.4f} "
        f"AUC={auc_lin:.4f} marge={marge_lin:.3f}"
    )
    print(
        f"  SVM RBF     : acc={acc_svm_emb:.4f} "
        f"AUC={auc_svm_emb:.4f} best={svm_emb.best_params_}"
    )
    print(
        f"  Ridge RBF   : acc={acc_kr_emb:.4f} "
        f"AUC={auc_kr_emb:.4f} best={kr_emb.best_params_}"
    )

# ── bilan ────────────────────────────────────────────────────────────────────
print("\n" + "=" * 72)
print("BILAN — ÉVOLUTION PAR COUCHE")
print("=" * 72)

print(
    f"  Ambient QP deg<={params['deg_start']} : "
    f"acc={sol0['acc_te']:.4f}  "
    f"AUC={sol0['auc']:.4f}  "
    f"‖u‖_H={sol0['normH']:.4f}"
)

print("\n  Couche | dim | QP acc | QP AUC | SVM acc | SVM AUC | Ridge acc | Ridge AUC")
print("  " + "-" * 78)

for r in layer_results:
    print(
        f"    {r['layer']:2d}   |"
        f" {r['dim']:3d} |"
        f" {r['qp_acc']:.4f} |"
        f" {r['qp_auc']:.4f} |"
        f" {r['svm_acc']:.4f} |"
        f" {r['svm_auc']:.4f} |"
        f"   {r['ridge_acc']:.4f} |"
        f"   {r['ridge_auc']:.4f}"
    )

print("\n  Références ambient :")
print(
    f"  SVM RBF   : acc={acc_svm:.4f} AUC={auc_svm:.4f}"
)
print(
    f"  Ridge RBF : acc={acc_kr:.4f} AUC={auc_kr:.4f}"
)

digits [3, 5] vs [8], 4×4 -> dim 16
  train: 300  test: 1000  unlabel`ed: 3000

Pool maître : 152 monômes (actifs deg<=2)
Construction du Gram maître PAR MOMENTS pour ordres [1, 2] ...
Gram maître prêt.

QP AMBIANT deg<=2
  [QP deg<=2] n_feat=152 rang=152 | faisable=True marge=1.000 ‖u‖_H=21.9413 MSE_te=21.6743 | acc_tr=1.000 acc_te=0.8700 acc_te_seuil=0.8490 (thr=-0.6623) AUC=0.9230

PLONGEMENT SPECTRAL (1 couche(s), embed_dim=152, k_proj=50, n_deg=2, bande=[0.001,2])
── couche 1 ──
  monômes deg 1..2 dans R^16
  n_feat = 152
  précalcul des moments jusqu'au degré 4 ...
  4845 moments non nuls (potentiels)
  spectre brut : min=0.258465 max=8.53615 median=1.66584
  VP < 1 : 23  | VP < 0.5 : 2  | VP < 0.1 : 0
  spectre : 152 VP | dans la bande : 152 | retenues (top AUC train) : 152
  VP gardées (λ) : [1.     1.     1.     1.3278 5.8184 1.     1.9861 5.763  1.7312 1.895  1.8333 5.2472 4.2386 2.2539
 2.0982 3.9927 1.7978 6.2637 7.0418 3.7924 1.     2.7509 0.878  3.0914 1.6708 3.0261 1.   